# Holo Quant — quantitative finance, 100% in your browser

A Pyodide-runnable quant stack, all offline & content-addressed: **statsmodels** (econometrics), **cvxpy + clarabel** (portfolio optimization), **mplfinance** (candlesticks), **backtrader** (backtesting), on numpy/scipy/pandas/scikit-learn. No server, no data feed — synthetic data so it runs anywhere. Run all cells.

In [ ]:
# Offline setup: piplite-index wheels (backtrader, mplfinance) + dist packages (no network).
import piplite
await piplite.install(['backtrader', 'mplfinance'])
import pyodide_js
await pyodide_js.loadPackage(['statsmodels', 'cvxpy-base', 'clarabel', 'scikit-learn'])
print('quant stack ready (offline)')

## 1 · Econometrics — CAPM α/β by OLS (statsmodels)

In [ ]:
import numpy as np, statsmodels.api as sm
rng = np.random.default_rng(1)
mkt = rng.normal(0.0005, 0.01, 500)                              # market daily returns
asset = 0.0002 + 1.3*mkt + rng.normal(0, 0.005, 500)            # true alpha=2bps, beta=1.3
res = sm.OLS(asset, sm.add_constant(mkt)).fit()
print(f'alpha (daily) = {res.params[0]:+.5f}')
print(f'beta          = {res.params[1]:.3f}')
print(f'R^2           = {res.rsquared:.3f}')

## 2 · Portfolio optimization — min-variance at a target return (cvxpy + clarabel)
Convex mean–variance optimization, solved on the GPU-era WASM build by the clarabel solver.

In [ ]:
import cvxpy as cp, numpy as np
mu = np.array([0.10, 0.12, 0.08, 0.15])                          # expected annual returns
S = np.array([[0.040,0.006,0.000,0.004],
              [0.006,0.090,0.000,0.010],
              [0.000,0.000,0.020,0.002],
              [0.004,0.010,0.002,0.120]])                        # covariance
w = cp.Variable(4); target = 0.11
prob = cp.Problem(cp.Minimize(cp.quad_form(w, cp.psd_wrap(S))),
                  [cp.sum(w) == 1, w >= 0, mu @ w >= target])
prob.solve(solver=cp.CLARABEL)
print('solver        :', prob.solver_stats.solver_name)
print('weights       :', np.round(w.value, 3))
print('exp. return   :', round(float(mu @ w.value), 4))
print('risk (std)    :', round(float(np.sqrt(w.value @ S @ w.value)), 4))

## 3 · Candlesticks with moving averages (mplfinance)

In [ ]:
import numpy as np, pandas as pd, mplfinance as mpf
rng = np.random.default_rng(2); n = 60
idx = pd.date_range('2026-01-01', periods=n, freq='D')
close = 100 + np.cumsum(rng.normal(0, 1, n))
op = close + rng.normal(0, 0.5, n)
hi = np.maximum(op, close) + np.abs(rng.normal(0, 0.8, n))
lo = np.minimum(op, close) - np.abs(rng.normal(0, 0.8, n))
df = pd.DataFrame({'Open':op,'High':hi,'Low':lo,'Close':close,'Volume':rng.integers(1000,5000,n)}, index=idx)
mpf.plot(df, type='candle', volume=True, mav=(5, 20), style='charles', figsize=(9,5))

## 4 · Backtest an SMA crossover (backtrader)

In [ ]:
import backtrader as bt, numpy as np, pandas as pd
class SmaCross(bt.Strategy):
    params = dict(p1=5, p2=20)
    def __init__(self):
        self.cross = bt.ind.CrossOver(bt.ind.SMA(period=self.p.p1), bt.ind.SMA(period=self.p.p2))
    def next(self):
        if not self.position and self.cross > 0: self.buy()
        elif self.position and self.cross < 0: self.close()
rng = np.random.default_rng(3); n = 300
idx = pd.date_range('2025-01-01', periods=n, freq='D')
price = 100 + np.cumsum(rng.normal(0.05, 1, n))
df = pd.DataFrame({'open':price,'high':price+1,'low':price-1,'close':price,'volume':1000}, index=idx)
cer = bt.Cerebro(); cer.broker.setcash(10000.0)
cer.adddata(bt.feeds.PandasData(dataname=df)); cer.addstrategy(SmaCross)
start = cer.broker.getvalue(); cer.run(); end = cer.broker.getvalue()
print(f'start  ${start:,.2f}')
print(f'end    ${end:,.2f}')
print(f'return {(end/start-1)*100:+.1f}%')

### What's included (all offline, content-addressed)
- **numpy · scipy · pandas · scikit-learn** — the compute + ML foundation (Pyodide dist).
- **statsmodels** — regression, ARIMA, time-series econometrics.
- **cvxpy + clarabel** — convex/portfolio optimization that actually solves in-browser.
- **mplfinance** — OHLC / candlestick charts.
- **backtrader** — event-driven strategy backtesting.

Flagships with native cores that have no WASM build (QuantLib, TA-Lib, vectorbt/numba, Qiskit) are deliberately omitted — they can't run 100% in-browser. cvxpy covers portfolio optimization directly.